In [1]:
import tensorflow as tf

import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

In [2]:
IMG_SIZE=224

BATCH_SIZE=32
EPOCHS=10
N_TRAIN=4000
N_TEST=800

NUM_CLASSES=20

VOC_LABELS=['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
            'diningtable', 'dog', 'horse', 'motorbike', 'person',   'sheep', 'sofa', 'train', 'tvmonitor']

In [3]:
ds_train_full,ds_val_full=tfds.load('voc/2007',split=['train+validation','test'], as_supervised=False)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating test examples...: 0 examples [00:00, ? examples/s]

...

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/voc/2007/incomplete.0LTK5V_5.0.0/voc-train.tfrecord*...:   0%|          | …

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/voc/2007/incomplete.0LTK5V_5.0.0/voc-validation.tfrecord*...:   0%|       …

Dataset voc downloaded and prepared to /root/tensorflow_datasets/voc/2007/5.0.0. Subsequent calls will reuse this data.


In [4]:
def has_object(sample):
    n=tf.shape(sample['objects']['bbox'])[0]

    return n>0

In [5]:
def preprocess_sample(sample):
    img=sample['image']
    bboxes=sample['objects']['bbox'] #yxyx
    labels=sample['objects']['label']

    img=tf.image.resize(img,(IMG_SIZE,IMG_SIZE))
    img=img/255.0

    xmin=bboxes[:,1]
    ymin=bboxes[:,0]
    xmax=bboxes[:,3]
    ymax=bboxes[:,2]

    bboxes_xyxy=tf.stack([xmin,ymin,xmax,ymax],axis=-1)

    h=ymax-ymin
    w=xmax-xmin
    area=h*w

    idx=tf.argmax(area)

    bbox_target=bboxes_xyxy[idx]
    label_target=labels[idx]
    return img,{'classification_output':label_target,'bbox_output':bbox_target,'all_labels':labels,'all_bboxes':bboxes_xyxy}


In [6]:
ds_train_full=ds_train_full.filter(has_object)

In [7]:
ds_train_full=ds_train_full.map(preprocess_sample).shuffle(1000)
# ds_train.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [8]:
ds_train=ds_train_full.map(lambda img,t: (img,{'classification_output':t['classification_output'],'bbox_output':t['bbox_output']})).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [9]:
ds_val_full=ds_val_full.filter(has_object)

In [10]:
ds_val_full=ds_val_full.map(preprocess_sample)

In [11]:
ds_val_full=ds_val_full.shuffle(1000)

In [12]:
ds_val=ds_val_full.map(lambda img,t: (img,{'classification_output':t['classification_output'],'bbox_output':t['bbox_output']})).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [13]:
backbone=tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE,IMG_SIZE,3),include_top=False,weights='imagenet')

In [14]:
backbone.trainable=False

In [15]:
inputs=tf.keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))

x=backbone(inputs)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
x=tf.keras.layers.Dense(128,activation='relu')(x)

classification_output=tf.keras.layers.Dense(NUM_CLASSES,activation='softmax',name='classification_output')(x)

bbox_output=tf.keras.layers.Dense(4,activation='sigmoid',name='bbox_output')(x)

model=tf.keras.Model(inputs=inputs,outputs=[classification_output,bbox_output])

In [16]:
def iou_metric(y_true,y_pred):
    x1_t=y_true[:,0]
    y1_t=y_true[:,1]
    x2_t=y_true[:,2]
    y2_t=y_true[:,3]

    x1_p=y_pred[:,0]
    y1_p=y_pred[:,1]
    x2_p=y_pred[:,2]
    y2_p=y_pred[:,3]

    x1_inter=tf.maximum(x1_t,x1_p)
    y1_inter=tf.maximum(y1_t,y1_p)
    x2_inter=tf.minimum(x2_t,x2_p)
    y2_inter=tf.minimum(y2_t,y2_p)

    inter_width=tf.maximum(0.0,x2_inter-x1_inter)
    inter_height=tf.maximum(0.0,y2_inter-y1_inter)
    inter_area=inter_width*inter_height

    true_area=(x2_t-x1_t)*(y2_t-y1_t)
    pred_area=(x2_p-x1_p)*(y2_p-y1_p)   
    union_area=true_area+pred_area-inter_area
    iou=inter_area/(union_area+1e-6)
    return tf.reduce_mean(iou)


In [ ]:
model.compile(optimizer='adam',
              loss={'classification_output':'sparse_categorical_crossentropy',
                    'bbox_output':'mse'},
                    metrics={'classification_output':'accuracy','bbox_output':iou_metric})

: 

In [18]:
history=model.fit(ds_train,validation_data=ds_val,epochs=EPOCHS)    

Epoch 1/10
    157/Unknown 65s 230ms/step - bbox_output_iou_metric: 0.3056 - bbox_output_loss: 0.0667 - classification_output_accuracy: 0.5582 - classification_output_loss: 1.5831 - loss: 1.6498

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


157/157 ━━━━━━━━━━━━━━━━━━━━ 114s 545ms/step - bbox_output_iou_metric: 0.3059 - bbox_output_loss: 0.0666 - classification_output_accuracy: 0.5589 - classification_output_loss: 1.5803 - loss: 1.6469 - val_bbox_output_iou_metric: 0.3958 - val_bbox_output_loss: 0.0439 - val_classification_output_accuracy: 0.7619 - val_classification_output_loss: 0.7652 - val_loss: 0.8091
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 55s 127ms/step - bbox_output_iou_metric: 0.3752 - bbox_output_loss: 0.0406 - classification_output_accuracy: 0.8072 - classification_output_loss: 0.6155 - loss: 0.6561 - val_bbox_output_iou_metric: 0.3905 - val_bbox_output_loss: 0.0361 - val_classification_output_accuracy: 0.7631 - val_classification_output_loss: 0.7579 - val_loss: 0.7946
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 41s 250ms/step - bbox_output_iou_metric: 0.3964 - bbox_output_loss: 0.0346 - classification_output_accuracy: 0.8583 - classification_output_loss: 0.4411 - loss: 0.4757 - val_bbox_output_iou_metric: 0.3980 - v

: 

: 